# Utility comparison on multi-MNIST (1-hidden-layer LTU)

Train a 2-layer LTU network (input → hidden LTU → output, with each hidden unit
hardwired to one output) on 2-task multi-MNIST and track four per-input-weight
EMA utilities over training. **No pruning** — `M_in` is all ones; this is
purely observational.

The four utilities, all on input-layer weights `W_in[j, h]`:
- **Contribution**: `|x_j| · |W_in[j, h]|`.
- **Weight magnitude**: `|W_in[j, h]|`.
- **Softmax-CE flip → LTU target → BCE LOO + informative mask**: per-input-weight
  BCE LOO against `target_h` (binary), gated by `informative_h`.
- **Softmax-CE flip → LTU target → L1 LOO + informative mask, preactivation target**:
  per-input-weight L1 LOO `u = |e+c| − |e|` with `e = target_h − z1` (raw preact,
  not sigmoid), gated by `informative_h`.

Plots (mirroring the linear notebook in this directory):
- Loss over time.
- Avg utility over time, split by within-task vs cross-task.
- Final distributions, split by within/cross.
- Separation metric: # connections in the [P2 within, P98 cross] overlap region.

In [5]:
import os
import sys

# Make the repo's `data` module importable. `data.py` lives in
# phd/structure_search/, so we need that directory on the path.
REPO_ROOT = '/home/edan/local_projects/phd_research'
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'phd', 'structure_search')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from phd.jax_core.models import ltu

# Multi-MNIST 2-task layout
N_TASKS = 2
NUM_CLASSES = 10
INPUT_PER_TASK = 784
INPUT_DIM = INPUT_PER_TASK * N_TASKS              # 1568
OUTPUT_DIM = NUM_CLASSES * N_TASKS                # 20

# 2-layer LTU layout — each hidden unit hardwired to one output.
N_HIDDEN = 100
HIDDEN_PER_OUTPUT = N_HIDDEN // OUTPUT_DIM        # 5

print('JAX device:', jax.devices()[0])
print(f'INPUT_DIM={INPUT_DIM}  N_HIDDEN={N_HIDDEN}  OUTPUT_DIM={OUTPUT_DIM}  HIDDEN_PER_OUTPUT={HIDDEN_PER_OUTPUT}')

JAX device: cuda:0
INPUT_DIM=1568  N_HIDDEN=100  OUTPUT_DIM=20  HIDDEN_PER_OUTPUT=5


## Data loading

In [6]:
def load_data():
    """Load MNIST and standardize per-pixel (mean 0, std 1).

    Returns (images, labels) as JAX arrays. Each image is shape (784,).
    Compose two of these to make a multi-MNIST sample.
    """
    from data import load_dataset
    images, labels, _, _ = load_dataset('mnist', split='train')
    images = np.asarray(images, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    mean = images.mean(axis=0, keepdims=True)
    std = images.std(axis=0, keepdims=True)
    normalized = (images - mean) / np.maximum(std, 1e-3)
    return jnp.asarray(normalized), jnp.asarray(labels)


images, labels = load_data()
print('images:', images.shape, 'dtype', images.dtype)
print('labels:', labels.shape, 'min/max:', int(labels.min()), int(labels.max()))

images: (60000, 784) dtype float32
labels: (60000,) min/max: 0 9


## Architecture helpers

In [7]:
def hidden_to_output_map():
    """Sequential routing: hidden unit i → output i // HIDDEN_PER_OUTPUT."""
    return jnp.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT


def hidden_unit_task_ids():
    """Task identity of each hidden unit (derived from its assigned output)."""
    return (jnp.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT) // NUM_CLASSES


def forward(W_in, W_out, x):
    """2-layer forward.
    W_in: (INPUT_DIM, N_HIDDEN). W_out: (N_HIDDEN, OUTPUT_DIM).
    Returns (logits, h, z1)."""
    z1 = x @ W_in                                             # (N_HIDDEN,)
    h = ltu(z1)                                               # binary {0,1} forward, sigmoid-STE backward
    logits = h @ W_out                                        # (OUTPUT_DIM,)
    return logits, h, z1


def loss_fn(W_in, W_out, x, y):
    """Per-task softmax CE summed across tasks."""
    logits, _, _ = forward(W_in, W_out, x)
    logits_per_task = logits.reshape(N_TASKS, NUM_CLASSES)
    lp = jax.nn.log_softmax(logits_per_task, axis=-1)
    return -jnp.mean(jnp.sum(jax.nn.one_hot(y, NUM_CLASSES) * lp, axis=-1))

## Utility functions

All return shape `(INPUT_DIM, N_HIDDEN)`. Sign convention: **U > 0 ⇒ removing the weight
hurts loss ⇒ keep**. Edit freely.

In [8]:
def contribution_utility(x, W_in):
    """|x_j| * |W_in[j, h]|: instantaneous magnitude of this weight's contribution to z1."""
    return jnp.abs(x[:, None]) * jnp.abs(W_in)


def weight_magnitude_utility(W_in):
    """|W_in[j, h]|."""
    return jnp.abs(W_in)


def hidden_flip_utility(h, W_out, logits, y):
    """Per-hidden-unit ΔCE if we flip h_i 0↔1, summed across tasks.

    For each unit i, compute the new task-logits if we flip h_i (i.e., add
    `flip_i * W_out[i, :]` to the logits, where flip_i = 1 - 2*h_i so it's
    -1 when h=1 and +1 when h=0). Compare per-task softmax CE before/after.
    Positive U_flip[i] = flipping hurts → keep current state. Negative = flip helps.
    """
    logits_pt = logits.reshape(N_TASKS, NUM_CLASSES)               # (T, C)
    flip = 1.0 - 2.0 * h                                           # (HIDDEN,)
    W_out_pt = W_out.reshape(N_HIDDEN, N_TASKS, NUM_CLASSES)        # (H, T, C)
    logits_flip = logits_pt[None, :, :] + flip[:, None, None] * W_out_pt  # (H, T, C)
    lp_pt   = jax.nn.log_softmax(logits_pt,   axis=-1)
    lp_flip = jax.nn.log_softmax(logits_flip, axis=-1)
    nll_pt   = -lp_pt[jnp.arange(N_TASKS), y]                      # (T,)
    nll_flip = -lp_flip[:, jnp.arange(N_TASKS), y]                 # (H, T)
    return (nll_flip - nll_pt[None, :]).sum(axis=-1)               # (HIDDEN,)


def ltu_targets(h, U_flip):
    """Derive binary target_h and informative gate.

    target_h = 1-h if flipping helps (U_flip < 0) else h.
    informative = (h > 0) | (target_h > 0): True if currently active OR should be.
    """
    should_flip = U_flip < 0.0
    target_h = jnp.where(should_flip, 1.0 - h, h)
    informative = (h > 0.0) | (target_h > 0.0)
    return target_h, informative.astype(jnp.float32)


def w_in_bce_informative_utility(x, W_in, preact, target_h, informative):
    """Per-input-weight BCE LOO, gated by informative_h.

    Treats z1 (preact) as a logit, target_h as binary label. ΔBCE if we
    remove W_in[j, h]: preact_wo = preact - x_j*W_in[j,h]; recompute BCE.
    Positive U = removing increases BCE = keep.
    """
    c = W_in * x[:, None]                                          # (IN, HIDDEN)
    preact_wo = preact[None, :] - c                                # (IN, HIDDEN)
    lp = (target_h * jax.nn.log_sigmoid(preact)
          + (1.0 - target_h) * jax.nn.log_sigmoid(-preact))         # (HIDDEN,)
    lp_wo = (target_h[None, :] * jax.nn.log_sigmoid(preact_wo)
             + (1.0 - target_h)[None, :] * jax.nn.log_sigmoid(-preact_wo))
    U = (-lp_wo) - (-lp[None, :])                                  # ΔBCE
    return U * informative[None, :]


def w_in_l1_informative_utility(x, W_in, preact, target_h, informative):
    """Per-input-weight L1 LOO, e = target_h - z1 (preact, NOT sigmoid).

    u = |e + c| - |e|, where c = x_j * W_in[j, h]. Positive U = keep.
    """
    c = W_in * x[:, None]                                          # (IN, HIDDEN)
    e = target_h - preact                                          # (HIDDEN,)
    U = jnp.abs(e[None, :] + c) - jnp.abs(e[None, :])
    return U * informative[None, :]


UTILITY_KEYS = ['U_contribution', 'U_weight_magnitude',
                'U_bce_informative', 'U_l1_informative']

## Training function

Edit freely. JIT-compatible — wrapped with `jax.jit` in the next cell. Online SGD
batch=1 on randomly-sampled multi-MNIST pairs.

In [9]:
def train(W_in_init, W_out_init, W_out_mask, images, labels, *,
          lr=2**-5,
          beta=0.998,
          n_steps=225_000,
          snapshot_every=1000,
          permute_period=0,
          seed=0):
    """Train a 2-layer LTU network on multi-MNIST and snapshot all four utility EMAs.

    Args:
      W_in_init: initial input weights, shape (INPUT_DIM, N_HIDDEN).
      W_out_init: initial output weights, shape (N_HIDDEN, OUTPUT_DIM); only the
        per-row entry where the routing mask is 1 is trainable.
      W_out_mask: (N_HIDDEN, OUTPUT_DIM) one-hot mask. Gradients on W_out are
        multiplied by this so the routing stays fixed.
      images, labels: standardized MNIST.
      lr: SGD learning rate.
      beta: EMA decay for the four utilities.
      n_steps: total training steps.
      snapshot_every: cadence of utility-EMA snapshots (also avg-loss bins).
      permute_period: every N steps, permute one task's class labels. 0 = stationary.
      seed: PRNG seed.

    Returns:
      dict with keys:
        'step', 'avg_loss': (n_snapshots,)
        'W_in', 'W_out':    (n_snapshots, ...) snapshots
        'U_<name>':         (n_snapshots, IN, HIDDEN) bias-corrected EMA
        'final_W_in', 'final_W_out': final state
    """
    n_snapshots = n_steps // snapshot_every
    perm0 = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1 = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    U_contrib = jnp.zeros_like(W_in_init)
    U_mag     = jnp.zeros_like(W_in_init)
    U_bce     = jnp.zeros_like(W_in_init)
    U_l1      = jnp.zeros_like(W_in_init)

    init_carry = (W_in_init, W_out_init,
                  U_contrib, U_mag, U_bce, U_l1,
                  perm0, perm1, jnp.array(0, dtype=jnp.int32))

    def make_sample(key):
        k1, k2 = jax.random.split(key)
        idx1 = jax.random.randint(k1, (), 0, images.shape[0])
        idx2 = jax.random.randint(k2, (), 0, images.shape[0])
        x = jnp.concatenate([images[idx1], images[idx2]])
        y_raw = jnp.array([labels[idx1], labels[idx2]])
        return x, y_raw

    def step_fn(carry, key):
        (W_in, W_out, U_contrib, U_mag, U_bce, U_l1, perm0, perm1, t) = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])

        # Loss + grads (W_out gradient gets masked to the routing).
        loss, (g_in, g_out) = jax.value_and_grad(loss_fn, argnums=(0, 1))(
            W_in, W_out, x, y)

        # Forward state for utilities (uses pre-update W).
        logits, h, z1 = forward(W_in, W_out, x)

        # Instantaneous utilities.
        u_contrib = contribution_utility(x, W_in)
        u_mag     = weight_magnitude_utility(W_in)
        U_flip    = hidden_flip_utility(h, W_out, logits, y)
        target_h, informative = ltu_targets(h, U_flip)
        u_bce = w_in_bce_informative_utility(x, W_in, z1, target_h, informative)
        u_l1  = w_in_l1_informative_utility(x, W_in, z1, target_h, informative)

        # EMA updates.
        U_contrib = beta * U_contrib + (1.0 - beta) * u_contrib
        U_mag     = beta * U_mag     + (1.0 - beta) * u_mag
        U_bce     = beta * U_bce     + (1.0 - beta) * u_bce
        U_l1      = beta * U_l1      + (1.0 - beta) * u_l1

        # SGD update — W_out gradient masked to keep the 1-to-1 routing.
        W_in  = W_in  - lr * g_in
        W_out = W_out - lr * g_out * W_out_mask
        t = t + 1

        # Optional permutation.
        if permute_period > 0:
            should_perm = (t >= permute_period) & (t % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1 = jnp.where(should_perm & (which == 1), new_perm, perm1)

        return (W_in, W_out, U_contrib, U_mag, U_bce, U_l1, perm0, perm1, t), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        (W_in, W_out, U_contrib, U_mag, U_bce, U_l1, _, _, t) = carry
        bias_corr = 1.0 / jnp.maximum(1.0 - beta ** t.astype(jnp.float32), 1e-12)
        snap = dict(
            step=t,
            avg_loss=losses.mean(),
            W_in=W_in,
            W_out=W_out,
            U_contribution=U_contrib * bias_corr,
            U_weight_magnitude=U_mag * bias_corr,
            U_bce_informative=U_bce * bias_corr,
            U_l1_informative=U_l1 * bias_corr,
        )
        return carry, snap

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_snapshots)
    final_carry, snaps_hist = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps_hist.items()}
    snaps['final_W_in']  = jax.device_get(final_carry[0])
    snaps['final_W_out'] = jax.device_get(final_carry[1])
    return snaps

## Run training

The first run JIT-compiles (slow); subsequent runs with the same `n_steps`,
`snapshot_every`, `permute_period` reuse the cache.

In [10]:
train_jit = jax.jit(
    train,
    static_argnames=('n_steps', 'snapshot_every', 'permute_period', 'lr', 'beta', 'seed'),
)

# W_out routing: each hidden unit drives one output via a one-hot row.
h2o = hidden_to_output_map()
W_out_mask = jax.nn.one_hot(h2o, OUTPUT_DIM)            # (N_HIDDEN, OUTPUT_DIM)

# Init weights. W_in: small Kaiming-style. W_out: small random scaled by mask.
init_key = jax.random.key(0)
k_in, k_out = jax.random.split(init_key)
w_in_bound  = jnp.sqrt(3.0 / float(INPUT_DIM))
W_in_init  = jax.random.uniform(k_in,  (INPUT_DIM, N_HIDDEN),
                                minval=-w_in_bound, maxval=w_in_bound)
w_out_bound = jnp.sqrt(3.0 / float(HIDDEN_PER_OUTPUT))
W_out_init = jax.random.uniform(k_out, (N_HIDDEN, OUTPUT_DIM),
                                minval=-w_out_bound, maxval=w_out_bound) * W_out_mask

snaps = train_jit(
    W_in_init, W_out_init, W_out_mask, images, labels,
    lr=2**-5,
    beta=0.998,
    n_steps=225_000,
    snapshot_every=1_000,
    permute_period=0,
    seed=0,
)

print('snapshots:', len(snaps['avg_loss']))
print('avg_loss start/end:', float(snaps['avg_loss'][0]), '/', float(snaps['avg_loss'][-1]))

snapshots: 225
avg_loss start/end: 1.6382375955581665 / 0.3065308928489685


## Plotting

In [11]:
# Cross-task / within-task masks for input weights (IN, HIDDEN).
input_task = np.arange(INPUT_DIM) // INPUT_PER_TASK                      # (IN,)
hidden_task = (np.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT) // NUM_CLASSES   # (HIDDEN,)
within_mask = (input_task[:, None] == hidden_task[None, :])              # (IN, HIDDEN) bool
cross_mask = ~within_mask
print(f'within-task connections: {within_mask.sum()}, cross-task: {cross_mask.sum()}')

WITHIN_COLOR = '#1f77b4'   # blue
CROSS_COLOR  = '#d62728'   # red

within-task connections: 78400, cross-task: 78400


In [12]:
def plot_avg_utility_over_time(snaps, utility_keys=UTILITY_KEYS):
    """For each utility, plot the mean over within-task and cross-task connections."""
    steps = np.asarray(snaps['step']) if 'step' in snaps else np.arange(len(snaps['avg_loss']))
    fig = make_subplots(rows=1, cols=len(utility_keys), subplot_titles=utility_keys,
                        shared_yaxes=False)
    for i, name in enumerate(utility_keys, start=1):
        U = np.asarray(snaps[name])  # (n_snap, IN, HIDDEN)
        within_mean = U[:, within_mask].mean(axis=-1)
        cross_mean  = U[:, cross_mask].mean(axis=-1)
        showlegend = (i == 1)
        fig.add_trace(go.Scatter(
            x=steps, y=within_mean, mode='lines', name='within-task',
            line=dict(color=WITHIN_COLOR), showlegend=showlegend,
            legendgroup='within'), row=1, col=i)
        fig.add_trace(go.Scatter(
            x=steps, y=cross_mean, mode='lines', name='cross-task',
            line=dict(color=CROSS_COLOR), showlegend=showlegend,
            legendgroup='cross'), row=1, col=i)
        fig.update_xaxes(title_text='step', row=1, col=i)
    fig.update_yaxes(title_text='mean utility', row=1, col=1)
    fig.update_layout(title='Average utility over time (per task type)',
                      height=420, width=320 * len(utility_keys))
    fig.show()
    return fig

In [13]:
def plot_final_distributions(snaps, utility_keys=UTILITY_KEYS, bins=80):
    """Histograms of per-connection utility at the final snapshot, split within/cross.

    X-axis bounds: inner-95% range (P2.5 to P97.5 over all weights), extended by
    10% of the inner range on each side. Outliers beyond are clipped.
    """
    fig = make_subplots(rows=1, cols=len(utility_keys), subplot_titles=utility_keys,
                        shared_yaxes=False)
    for i, name in enumerate(utility_keys, start=1):
        U_final = np.asarray(snaps[name][-1])
        within_vals = U_final[within_mask]
        cross_vals  = U_final[cross_mask]
        all_vals = np.concatenate([within_vals, cross_vals])
        lo = np.percentile(all_vals, 2.5)
        hi = np.percentile(all_vals, 97.5)
        width = hi - lo
        if width <= 0:
            width = max(abs(lo), abs(hi), 1.0) * 0.1
        xmin = lo - 0.1 * width
        xmax = hi + 0.1 * width
        bin_size = (xmax - xmin) / bins
        showlegend = (i == 1)
        fig.add_trace(go.Histogram(
            x=cross_vals, opacity=0.55, name='cross-task',
            marker_color=CROSS_COLOR,
            xbins=dict(start=xmin, end=xmax, size=bin_size),
            showlegend=showlegend, legendgroup='cross'), row=1, col=i)
        fig.add_trace(go.Histogram(
            x=within_vals, opacity=0.55, name='within-task',
            marker_color=WITHIN_COLOR,
            xbins=dict(start=xmin, end=xmax, size=bin_size),
            showlegend=showlegend, legendgroup='within'), row=1, col=i)
        fig.update_xaxes(range=[xmin, xmax], title_text='utility', row=1, col=i)
    fig.update_yaxes(title_text='# connections', row=1, col=1)
    fig.update_layout(barmode='overlay', title='Final utility distributions',
                      height=420, width=320 * len(utility_keys))
    fig.show()
    return fig

In [14]:
def plot_separation_metric(snaps, utility_keys=UTILITY_KEYS, tail_pct=98):
    """Number of connections whose utility lies in the [P_{100-tail_pct} within,
    P_{tail_pct} cross] overlap region. Lower = cleaner separation, 0 = perfectly separated.
    """
    steps = np.asarray(snaps['step']) if 'step' in snaps else np.arange(len(snaps['avg_loss']))
    lo_pct = 100.0 - tail_pct
    fig = go.Figure()
    for name in utility_keys:
        U = np.asarray(snaps[name])
        cross_vals  = U[:, cross_mask]
        within_vals = U[:, within_mask]
        cross_high = np.percentile(cross_vals, tail_pct, axis=-1)
        within_low = np.percentile(within_vals, lo_pct, axis=-1)
        in_overlap = (U >= within_low[:, None, None]) & (U <= cross_high[:, None, None])
        counts = in_overlap.sum(axis=(1, 2))
        fig.add_trace(go.Scatter(x=steps, y=counts, mode='lines', name=name))
    fig.update_layout(
        title=f'Separation metric (tail_pct={tail_pct})',
        xaxis_title='step',
        yaxis_title=f'# connections in overlap [P{lo_pct:g} within, P{tail_pct:g} cross]',
        width=900, height=450,
    )
    fig.show()
    return fig


def plot_loss_over_time(snaps):
    """Per-snapshot mean training loss."""
    steps = np.asarray(snaps['step']) if 'step' in snaps else np.arange(len(snaps['avg_loss']))
    fig = go.Figure(go.Scatter(x=steps, y=np.asarray(snaps['avg_loss']),
                               mode='lines', name='loss'))
    fig.update_layout(title='Training loss',
                      xaxis_title='step',
                      yaxis_title='mean loss over snapshot interval',
                      width=800, height=380)
    fig.show()
    return fig

In [15]:
plot_loss_over_time(snaps)
plot_avg_utility_over_time(snaps)
plot_final_distributions(snaps)
plot_separation_metric(snaps)